# What is Convolution? (1D)

A **kernel** slides along a signal, computing dot products at each position.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11
torch.manual_seed(42)
np.random.seed(42)


## 1. 1D signal and kernel

In [ ]:
signal = torch.tensor([0., 1., 2., 3., 2., 1., 0., -1., -2., -1., 0.])
kernel = torch.tensor([1., 0., -1.])  # simple edge-like
print(f"signal length: {len(signal)}, kernel length: {len(kernel)}")

## 2. Manual convolution steps

In [ ]:
def conv1d_manual(x, k):
    out_len = len(x) - len(k) + 1
    out = []
    positions = []
    for i in range(out_len):
        patch = x[i:i+len(k)]
        out.append(float((patch * k).sum()))
        positions.append(i)
    return torch.tensor(out), positions

out, positions = conv1d_manual(signal, kernel)
print(f"output: {out}")

## 3. Plot signal and kernel

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 5))
axes[0].stem(range(len(signal)), signal.numpy(), linefmt='C0-', markerfmt='C0o', basefmt=' ')
axes[0].set_title('Input 1D signal'); axes[0].set_ylabel('value')
axes[1].stem(range(len(kernel)), kernel.numpy(), linefmt='C1-', markerfmt='C1o', basefmt=' ')
axes[1].set_title('Kernel [1, 0, -1]'); axes[1].set_xlabel('index')
plt.tight_layout(); plt.show()

## 4. Kernel sliding — multi-panel animation-style

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 5))
for i, pos in enumerate([0, 2, 4, 6]):
    patch = signal[pos:pos+3]
    axes[0, i].bar(range(3), patch.numpy(), color='steelblue')
    axes[0, i].set_title(f'patch @ pos {pos}'); axes[0, i].set_ylim(-3, 4)
    axes[1, i].bar([0], [out[pos].item()], color='coral', width=0.5)
    axes[1, i].set_title(f'output[{pos}]={out[pos]:.1f}'); axes[1, i].set_ylim(-3, 4)
plt.suptitle('Kernel sliding left → right'); plt.tight_layout(); plt.show()

## 5. Full output vs PyTorch conv1d

In [ ]:
pt_out = F.conv1d(signal.view(1, 1, -1), kernel.view(1, 1, -1)).squeeze()
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(out.numpy(), 'o-', label='manual', lw=2)
ax.plot(pt_out.numpy(), 'x--', label='F.conv1d', lw=2)
ax.set_xlabel('output index'); ax.legend()
ax.set_title('Convolution output')
plt.tight_layout(); plt.show()